# Pig Posture Recognition - ConvNeXt v3 (Generalization Fix)

**Kernprobleme des bisherigen Ansatzes:**
- Val-Split auf Instanz-Ebene = Data Leak (gleiche Bilder in Train+Val)
- Zu viel Hintergrund-Kontext (PAD_RATIO=0.25) = Kamera-spezifische Features
- Nur T2 = wenig Kamera-Diversitaet

**Aenderungen in diesem Notebook:**
1. GroupKFold nach `image_id` - kein Bild in Train UND Val
2. Camera-Level Split als ehrlichste Validierung
3. T1 + T2 kombiniert fuer maximale Kamera-Diversitaet
4. PAD_RATIO reduziert (0.10) - weniger Hintergrund
5. Staerkere geometrische Augmentierungen (RandomPerspective)
6. Val F1 wird jetzt ehrlich sein (~0.60-0.75) aber naeher am Kaggle-Score

## Configuration

In [1]:
# ============================================================
# CONFIGURATION
# ============================================================

DATA_ROOT = "/datasets/multi-view-pig-posture-recognition"

# --- Training Data: T1 + T2 combined ---
TRAIN_SETS = {
    "T1": {"csv": f"{DATA_ROOT}/train1.csv", "img_dir": f"{DATA_ROOT}/train1_images"},
    "T2": {"csv": f"{DATA_ROOT}/train2.csv", "img_dir": f"{DATA_ROOT}/train2_images"},
}

# Welche Sets zum Training verwenden?
# Option A: ["T1", "T2"]  -> Beide (mehr Kamera-Diversitaet, empfohlen)
# Option B: ["T2"]         -> Nur T2 (fuer T2_ Submission)
USE_SETS = ["T1", "T2"]

# Fuer Submission-Tag
TAG = "T2" if USE_SETS == ["T2"] else "T2"  # Kaggle Submission immer T2_

OUTPUT_DIR = f"runs/convnext_v3_{TAG.lower()}"

# Model
MODEL_NAME     = "convnext_base"
IMG_SIZE       = 288
BATCH_SIZE     = 32
EPOCHS         = 25
LR             = 3e-4
LABEL_SMOOTH   = 0.10
MIXUP_ALPHA    = 0.30
NUM_WORKERS    = 8
SEED           = 42
NUM_CLASSES    = 5

# WICHTIG: Reduziert von 0.25 auf 0.10
# Weniger Hintergrund = weniger Kamera-spezifische Features
PAD_RATIO      = 0.10

# Validation-Strategie:
# "image"  -> GroupKFold nach image_id (kein Bild-Leak, empfohlen)
# "camera" -> Leave-One-Camera-Out (haertester Test, nahe an Kaggle)
VAL_STRATEGY   = "image"
N_FOLDS        = 5

CLASS_NAMES = ["Lateral_lying_left", "Lateral_lying_right",
               "Sitting", "Standing", "Sternal_lying"]

print(f"Training Sets: {USE_SETS}")
print(f"Val Strategy: {VAL_STRATEGY}")
print(f"PAD_RATIO: {PAD_RATIO}")
print(f"Output: {OUTPUT_DIR}")

Training Sets: ['T1', 'T2']
Val Strategy: image
PAD_RATIO: 0.1
Output: runs/convnext_v3_t2


## Imports

In [2]:
!pip install timm tqdm scikit-learn pandas pillow matplotlib -q

In [3]:
import os, ast, re, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
from collections import Counter
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as Fn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import timm

from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.metrics import f1_score, classification_report

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPUs: {torch.cuda.device_count()}")

<jemalloc>: Unsupported system page size


Device: cuda
GPU: Tesla V100-SXM2-16GB
GPUs: 2


In [4]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Daten laden (T1 + T2 kombiniert)

In [5]:
dfs = []
for name in USE_SETS:
    cfg = TRAIN_SETS[name]
    tmp = pd.read_csv(cfg["csv"])
    tmp["img_dir"] = cfg["img_dir"]
    tmp["source"]  = name
    dfs.append(tmp)
    print(f"{name}: {len(tmp)} Instanzen, {tmp['image_id'].nunique()} Bilder")

df = pd.concat(dfs, axis=0, ignore_index=True)
print(f"\nGesamt: {len(df)} Instanzen, {df['image_id'].nunique()} unique Bilder")

# Kamera-ID extrahieren (z.B. pen1_orb_cam1)
df["camera_id"] = df["image_id"].apply(
    lambda x: re.search(r'(pen\d+_[a-zA-Z0-9]+_cam\d+)', str(x)).group(1)
    if re.search(r'(pen\d+_[a-zA-Z0-9]+_cam\d+)', str(x)) else "unknown"
)

print(f"\nKameras: {df['camera_id'].nunique()} Stueck")
print(df["camera_id"].value_counts().sort_index())

print(f"\nKlassenverteilung:")
for c in range(NUM_CLASSES):
    cnt = (df["class_id"] == c).sum()
    pct = 100 * cnt / len(df)
    bar = "#" * int(30 * cnt / len(df))
    print(f"  {c} - {CLASS_NAMES[c]:<22} {bar:<30} {cnt:>5}  ({pct:.1f}%)")

T1: 22934 Instanzen, 3090 Bilder
T2: 23450 Instanzen, 3150 Bilder

Gesamt: 46384 Instanzen, 3150 unique Bilder

Kameras: 8 Stueck
camera_id
pen1_orb_cam1     1444
pen1_orb_cam2     2976
pen1_tur_cam1      200
pen1_tur_cam2    16388
pen2_orb_cam1     5634
pen2_orb_cam2      120
pen2_tur_cam1    19426
pen2_tur_cam2      196
Name: count, dtype: int64

Klassenverteilung:
  0 - Lateral_lying_left     ###                             6136  (13.2%)
  1 - Lateral_lying_right    ####                            6811  (14.7%)
  2 - Sitting                                                1375  (3.0%)
  3 - Standing               ############                   19545  (42.1%)
  4 - Sternal_lying          ########                       12517  (27.0%)


## Validation-Split Analyse

Vergleich: Alter Split (Instanz-Level) vs. Neuer Split (Image/Camera-Level)

In [6]:
# Demonstriere das Data-Leak Problem
from sklearn.model_selection import train_test_split

# ALTER Split (Instanz-Level) - SO WIE BISHER
old_train, old_val = train_test_split(df, test_size=0.10, stratify=df["class_id"], random_state=SEED)

# Wie viele Val-Bilder tauchen auch in Train auf?
val_images = set(old_val["image_id"].unique())
train_images = set(old_train["image_id"].unique())
leaked_images = val_images & train_images

print("=== ALTER Split (Instanz-Level) ===")
print(f"Val-Bilder: {len(val_images)}")
print(f"Davon AUCH in Train: {len(leaked_images)} ({100*len(leaked_images)/len(val_images):.1f}%)")
print(f"-> {100*len(leaked_images)/len(val_images):.0f}% Data Leak!")
print()

# NEUER Split (Image-Level GroupKFold)
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
groups = df["image_id"].values

print(f"=== NEUER Split (GroupKFold nach image_id, {N_FOLDS} Folds) ===")
for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, df["class_id"], groups)):
    t_imgs = set(df.iloc[train_idx]["image_id"].unique())
    v_imgs = set(df.iloc[val_idx]["image_id"].unique())
    leak = t_imgs & v_imgs
    print(f"  Fold {fold+1}: Train={len(train_idx)}, Val={len(val_idx)}, "
          f"Bild-Leak={len(leak)} (0%)")

=== ALTER Split (Instanz-Level) ===
Val-Bilder: 2392
Davon AUCH in Train: 2392 (100.0%)
-> 100% Data Leak!

=== NEUER Split (GroupKFold nach image_id, 5 Folds) ===
  Fold 1: Train=37073, Val=9311, Bild-Leak=0 (0%)
  Fold 2: Train=37133, Val=9251, Bild-Leak=0 (0%)
  Fold 3: Train=37251, Val=9133, Bild-Leak=0 (0%)
  Fold 4: Train=37098, Val=9286, Bild-Leak=0 (0%)
  Fold 5: Train=36981, Val=9403, Bild-Leak=0 (0%)


## Dataset & Augmentierungen

Staerkere geometrische Augmentierungen fuer Perspektiv-Invarianz.

In [7]:
class PigPostureDataset(Dataset):
    def __init__(self, df, transform=None, pad_ratio=0.10):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(row["img_dir"], row["image_id"])
        img  = Image.open(path).convert("RGB")
        crop = self._crop(img, row["bbox"])
        if self.transform: crop = self.transform(crop)
        return crop, int(row["class_id"])


def get_train_transform(size=IMG_SIZE):
    return T.Compose([
        T.Resize((size + 48, size + 48)),
        T.RandomCrop(size),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.3),
        T.RandomRotation(degrees=30),
        # NEU: Perspektiv-Transformation fuer Kamera-Invarianz
        T.RandomPerspective(distortion_scale=0.3, p=0.5),
        T.RandomAffine(degrees=0, scale=(0.80, 1.20), shear=15),
        T.ColorJitter(brightness=0.6, contrast=0.6, saturation=0.5, hue=0.10),
        T.RandomGrayscale(p=0.15),
        T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        # Staerkeres Erasing - zwingt Modell, nicht auf einzelne Regionen zu bauen
        T.RandomErasing(p=0.4, scale=(0.02, 0.25)),
    ])

def get_val_transform(size=IMG_SIZE):
    return T.Compose([
        T.Resize((size, size)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

print("Dataset und Transforms definiert.")
print(f"PAD_RATIO={PAD_RATIO} (vorher 0.25 -> jetzt weniger Hintergrund)")

Dataset und Transforms definiert.
PAD_RATIO=0.1 (vorher 0.25 -> jetzt weniger Hintergrund)


## Helpers (MixUp, Train/Val Loop)

In [8]:
def mixup_data(x, y, alpha=0.3):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def train_one_epoch(model, loader, optimizer, scaler, criterion_plain):
    model.train()
    loss_sum, preds_all, labels_all = 0.0, [], []
    for imgs, labels in tqdm(loader, desc="  Train", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        imgs, y_a, y_b, lam = mixup_data(imgs, labels, alpha=MIXUP_ALPHA)
        optimizer.zero_grad()
        with autocast():
            logits = model(imgs)
            loss   = mixup_loss(criterion_plain, logits, y_a, y_b, lam)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(y_a.cpu().numpy())
    n = len(loader.dataset)
    return loss_sum / n, f1_score(labels_all, preds_all, average="macro", zero_division=0)


@torch.no_grad()
def validate_epoch(model, loader, criterion):
    model.eval()
    loss_sum, preds_all, labels_all = 0.0, [], []
    for imgs, labels in tqdm(loader, desc="  Val  ", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)
        loss_sum += loss.item() * imgs.size(0)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    n = len(loader.dataset)
    return loss_sum / n, f1_score(labels_all, preds_all, average="macro", zero_division=0), preds_all, labels_all

## K-Fold Training (GroupKFold nach image_id)

Kein Bild erscheint in Train UND Val gleichzeitig. Val F1 wird ehrlich sein.

In [9]:
print(f"Starte {N_FOLDS}-Fold GroupKFold (Gruppen: image_id)")
print(f"Datensatz: {len(df)} Instanzen, {df['image_id'].nunique()} Bilder")
print(f"Strategie: {VAL_STRATEGY}")
print()

if VAL_STRATEGY == "image":
    splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    split_groups = df["image_id"].values
elif VAL_STRATEGY == "camera":
    splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    split_groups = df["camera_id"].values

fold_results = []

for fold_idx, (train_idx, val_idx) in enumerate(splitter.split(df, df["class_id"], split_groups)):

    ckpt_path = os.path.join(OUTPUT_DIR, f"best_model_fold_{fold_idx+1}.pth")
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu")
        saved_f1 = ckpt.get("val_f1", 0)
        fold_results.append(saved_f1)
        print(f"Ueberspringe Fold {fold_idx+1}, existiert (val_f1={saved_f1:.4f})")
        continue

    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx + 1} / {N_FOLDS}")
    print(f"{'='*60}")

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx].reset_index(drop=True)

    # Pruefe: Kein Bild-Leak
    t_imgs = set(train_df["image_id"].unique())
    v_imgs = set(val_df["image_id"].unique())
    assert len(t_imgs & v_imgs) == 0, "BILD-LEAK ERKANNT!"

    # Zeige Kamera-Verteilung in Val
    val_cams = val_df["camera_id"].value_counts()
    print(f"Train: {len(train_df)} | Val: {len(val_df)}")
    print(f"Val Kameras: {dict(val_cams)}")

    train_ds = PigPostureDataset(train_df, transform=get_train_transform(), pad_ratio=PAD_RATIO)
    val_ds   = PigPostureDataset(val_df,   transform=get_val_transform(),   pad_ratio=PAD_RATIO)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    # Neues Modell pro Fold
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
    model = model.to(DEVICE)
    if torch.cuda.device_count() > 1:
        print(f"Nutze {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    # Class Weights
    counts  = Counter(train_df["class_id"].tolist())
    weights = torch.tensor(
        [len(train_df) / (NUM_CLASSES * max(counts.get(c, 1), 1)) for c in range(NUM_CLASSES)],
        dtype=torch.float32
    ).to(DEVICE)

    criterion       = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTH)
    criterion_plain = nn.CrossEntropyLoss(weight=weights)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler    = GradScaler()

    best_val_f1 = 0.0
    patience_counter = 0
    PATIENCE = 8

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_f1   = train_one_epoch(model, train_loader, optimizer, scaler, criterion_plain)
        val_loss, val_f1, _, _ = validate_epoch(model, val_loader, criterion)
        scheduler.step()
        lr = scheduler.get_last_lr()[0]

        improved = val_f1 > best_val_f1
        mark = "*" if improved else " "
        print(f"{mark} Fold {fold_idx+1} | Ep {epoch:02d}/{EPOCHS} | "
              f"Train L={train_loss:.4f} F1={train_f1:.4f} | "
              f"Val L={val_loss:.4f} F1={val_f1:.4f} | lr={lr:.2e}")

        if improved:
            best_val_f1 = val_f1
            patience_counter = 0
            state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save({
                "epoch": epoch, "model": state,
                "val_f1": val_f1, "model_name": MODEL_NAME,
                "val_strategy": VAL_STRATEGY, "pad_ratio": PAD_RATIO,
                "use_sets": USE_SETS,
            }, ckpt_path)
            print(f"  -> Gespeichert (val_f1={val_f1:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping nach {PATIENCE} Epochen ohne Verbesserung.")
                break

    fold_results.append(best_val_f1)
    print(f"\nFold {fold_idx+1} beendet! Bester Val F1: {best_val_f1:.4f}")

    # Aufraeumen
    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"{N_FOLDS}-Fold Training abgeschlossen!")
print(f"F1 Scores: {[round(x, 4) for x in fold_results]}")
print(f"Durchschnittlicher F1: {np.mean(fold_results):.4f} (+/- {np.std(fold_results):.4f})")
print(f"\nDieser Wert sollte NAEHER am Kaggle-Score sein als die bisherigen 0.93!")

Starte 5-Fold GroupKFold (Gruppen: image_id)
Datensatz: 46384 Instanzen, 3150 Bilder
Strategie: image

Ueberspringe Fold 1, existiert (val_f1=0.8016)
Ueberspringe Fold 2, existiert (val_f1=0.9166)

FOLD 3 / 5
Train: 37251 | Val: 9133
Val Kameras: {'pen2_tur_cam1': 3848, 'pen1_tur_cam2': 3170, 'pen2_orb_cam1': 1142, 'pen1_orb_cam2': 590, 'pen1_orb_cam1': 276, 'pen2_tur_cam2': 63, 'pen1_tur_cam1': 28, 'pen2_orb_cam2': 16}
Nutze 2 GPUs


  Train:   0%|          | 0/1164 [00:00<?, ?it/s]

  Val  :   0%|          | 0/286 [00:00<?, ?it/s]

* Fold 3 | Ep 01/25 | Train L=1.1753 F1=0.3701 | Val L=1.3099 F1=0.7001 | lr=2.99e-04
  -> Gespeichert (val_f1=0.7001)


  Train:   0%|          | 0/1164 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Classification Report (bestes Fold-Modell)

In [ ]:
# Lade bestes Fold und zeige detaillierten Report
best_fold = np.argmax(fold_results) + 1
print(f"Bestes Fold: {best_fold} (F1={fold_results[best_fold-1]:.4f})")

# Recreate val split fuer bestes fold
for fi, (train_idx, val_idx) in enumerate(splitter.split(df, df["class_id"], split_groups)):
    if fi == best_fold - 1:
        val_df = df.iloc[val_idx].reset_index(drop=True)
        break

val_ds = PigPostureDataset(val_df, transform=get_val_transform(), pad_ratio=PAD_RATIO)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

ckpt_path = os.path.join(OUTPUT_DIR, f"best_model_fold_{best_fold}.pth")
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
model.load_state_dict(ckpt["model"])
model.to(DEVICE).eval()

criterion = nn.CrossEntropyLoss()
_, _, val_preds, val_labels = validate_epoch(model, val_loader, criterion)
print(classification_report(val_labels, val_preds, target_names=CLASS_NAMES))
del model; torch.cuda.empty_cache()

## Inference & Submission

5-Fold Ensemble + TTA

In [ ]:
# === INFERENCE CONFIG ===
TEST_CSV    = f"{DATA_ROOT}/test.csv"
TEST_IMG_DIR = f"{DATA_ROOT}/test_images"
USE_TTA     = True
OUTPUT_FILE = f"{TAG}_convnext_v3_submission.csv"

CKPT_PATHS = [os.path.join(OUTPUT_DIR, f"best_model_fold_{i}.pth") for i in range(1, N_FOLDS+1)]
# Nur existierende Checkpoints
CKPT_PATHS = [p for p in CKPT_PATHS if os.path.exists(p)]
print(f"Checkpoints fuer Ensemble: {len(CKPT_PATHS)}")
for p in CKPT_PATHS:
    ckpt = torch.load(p, map_location="cpu")
    print(f"  {os.path.basename(p)} | val_f1={ckpt.get('val_f1', 0):.4f}")

In [ ]:
class PigTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.10):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        if self.transform: crop = self.transform(crop)
        return crop, row["row_id"]


S = IMG_SIZE
NORM = [[0.485,0.456,0.406],[0.229,0.224,0.225]]

TTA_TRANSFORMS = [
    T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(*NORM)]),
    T.Compose([T.Resize((S, S)), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(*NORM)]),
    T.Compose([T.Resize((S+32, S+32)), T.CenterCrop(S), T.ToTensor(), T.Normalize(*NORM)]),
    T.Compose([T.Resize((S+32, S+32)), T.CenterCrop(S), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(*NORM)]),
]
print(f"TTA Views: {len(TTA_TRANSFORMS)}")

In [ ]:
test_df = pd.read_csv(TEST_CSV)
print(f"Test instances: {len(test_df)}")

@torch.no_grad()
def predict_tta(model, df, img_dir, transforms):
    all_probs = []
    for i, tf in enumerate(transforms):
        ds     = PigTestDataset(df, img_dir, transform=tf, pad_ratio=PAD_RATIO)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        probs = []
        for imgs, _ in tqdm(loader, desc=f"  TTA {i+1}/{len(transforms)}", leave=False):
            with autocast():
                logits = model(imgs.to(DEVICE))
            probs.append(Fn.softmax(logits, dim=1).cpu().numpy())
        all_probs.append(np.vstack(probs))
    return np.mean(all_probs, axis=0)

transforms = TTA_TRANSFORMS if USE_TTA else [TTA_TRANSFORMS[0]]
ensemble_probs = []

for path in CKPT_PATHS:
    print(f"\nLade: {os.path.basename(path)}")
    ckpt = torch.load(path, map_location="cpu")
    name = ckpt.get("model_name", MODEL_NAME)
    print(f"  Model: {name}  |  Val F1: {ckpt.get('val_f1', 0):.4f}")
    model = timm.create_model(name, pretrained=False, num_classes=NUM_CLASSES)
    model.load_state_dict(ckpt["model"])
    model.to(DEVICE).eval()
    ensemble_probs.append(predict_tta(model, test_df, TEST_IMG_DIR, transforms))
    del model; torch.cuda.empty_cache()

predictions = np.mean(ensemble_probs, axis=0).argmax(axis=1)
print(f"\nDone: {len(predictions)} predictions aus {len(CKPT_PATHS)} Modellen")

In [ ]:
submission = pd.DataFrame({"row_id": test_df["row_id"].values, "class_id": predictions.astype(int)})
submission.to_csv(OUTPUT_FILE, index=False)
print(f"Gespeichert: {OUTPUT_FILE} ({len(submission)} Zeilen)")

assert list(submission.columns) == ["row_id", "class_id"]
assert set(submission["class_id"].unique()).issubset(set(range(5)))
assert len(submission) == len(test_df)
print("Sanity checks OK")

print("\nVorhergesagte Verteilung:")
counts = submission["class_id"].value_counts().sort_index()
for c in range(NUM_CLASSES):
    cnt = counts.get(c, 0)
    print(f"  {c} - {CLASS_NAMES[c]:<22} {cnt:>5}  ({100*cnt/len(submission):.1f}%)")

submission.head(10)

## Zusammenfassung

**Was dieses Notebook anders macht:**

| Aenderung | Vorher | Jetzt | Warum |
|---|---|---|---|
| Val-Split | Instanz-Level (Data Leak) | GroupKFold nach image_id | Keine gleichen Bilder in Train+Val |
| Training Data | Nur T2 | T1 + T2 | Mehr Kamera-Diversitaet |
| PAD_RATIO | 0.25 | 0.10 | Weniger Hintergrund-Kontext |
| Augmentierung | Standard | +RandomPerspective, +Shear, staerker | Perspektiv-Invarianz |
| Early Stopping | Nein | Ja (Patience=8) | Kein Overfitting |

**Erwartung:**
- Val F1 wird NIEDRIGER sein als vorher (0.70-0.85 statt 0.93)
- Kaggle F1 wird HOEHER sein als vorher (>0.62)
- Der Gap zwischen Val und Kaggle wird kleiner